# Clinical survival analyses

Adapted from the submitted repository, commit `fce9ade29ef6de731407a7fcf14098f30a272cd3`, for the public-data workflow. Current definitions and figure mappings are documented in `docs/`.


In [ ]:
import os
from pathlib import Path
from IPython.display import display
_candidates = [Path.cwd(), *Path.cwd().parents]
_project = next((p for p in _candidates if (p / "src/mitoxy/data.py").is_file()), None)
if _project is None:
    raise RuntimeError("Launch this notebook within the Mitoxy repository")
os.environ["MITOXY_PROJECT_ROOT"] = str(_project)


# Supplementary Figure S1 — clinical extension and survival sensitivity

All estimates are reconstructed from the public files recorded by the
paper-wide download notebook. The archived project was consulted only for
model definitions; it is not read by this notebook and no expected result
values are encoded.


In [ ]:
from pathlib import Path
import hashlib
import json
import re
import warnings

import numpy as np
import pandas as pd
import pyreadr
import statsmodels.api as sm
from scipy.stats import spearmanr

PROJECT_ROOT = Path(os.environ["MITOXY_PROJECT_ROOT"])
RAW_MANIFEST_PATH = (
    PROJECT_ROOT / "data" / "provenance" / "paper_public_source_manifest.csv"
)
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "figure_1_clean"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GENES = ["BAK1", "BAX", "BID", "SLC7A11"]
CORE3 = ["BAK1", "BAX", "BID"]
RANDOM_SEED = 20260913

raw_manifest = pd.read_csv(RAW_MANIFEST_PATH)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def source_path(dataset, role):
    matches = raw_manifest.loc[
        raw_manifest["dataset"].eq(dataset)
        & raw_manifest["role"].eq(role)
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one public source for ({dataset!r}, {role!r}); "
            f"found {len(matches)}"
        )
    row = matches.iloc[0]
    path = PROJECT_ROOT / str(row["relative_path"])
    if not path.exists():
        raise FileNotFoundError(path)
    if path.stat().st_size != int(row["size_bytes"]):
        raise RuntimeError(f"Source-size mismatch: {path}")
    if sha256_file(path) != str(row["sha256"]):
        raise RuntimeError(f"Source SHA-256 mismatch: {path}")
    return path


RAW = {
    "tcga_xena_expression": source_path(
        "TCGA-KIRC", "tumor RNA expression"
    ),
    "tcga_gdc_clinical_json": source_path(
        "TCGA-KIRC", "clinical API response"
    ),
    "cptac_rna": source_path(
        "CPTAC ccRCC", "tumor RNA expression"
    ),
    "cptac_protein": source_path(
        "CPTAC ccRCC", "tumor protein abundance"
    ),
    "cptac_clinical_xlsx": source_path(
        "CPTAC ccRCC", "clinical supplement"
    ),
    "emtab_expression": source_path(
        "E-MTAB-1980", "ccRCC_exp_log_quantile_normalized.txt"
    ),
    "emtab_clinical_xlsx": source_path(
        "E-MTAB-1980",
        "Sato_2013_Supplementary_Table_1_patient_characteristics.xlsx",
    ),
    "tracerx_tpm": source_path(
        "TRACERx Renal", "regional tumor TPM"
    ),
    "tracerx_annotation": source_path(
        "TRACERx Renal", "regional annotation"
    ),
    "tracerx_clinical": source_path(
        "TRACERx Renal", "clinical annotation"
    ),
}

print("Paper-wide public-source manifest loaded:", RAW_MANIFEST_PATH)
display(pd.DataFrame({"analysis_key": RAW.keys(), "path": RAW.values()}))


In [ ]:
def zscore(series):
    values = pd.to_numeric(series, errors="coerce")
    return (values - values.mean()) / values.std(ddof=0)


def equal4_score(frame):
    standardized = frame[GENES].apply(zscore, axis=0)
    return standardized.mean(axis=1)


def parse_grade(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"(?:G(?:RADE)?\s*)?([1-4])", str(value), flags=re.I)
    return float(match.group(1)) if match else np.nan


def parse_stage(value):
    if pd.isna(value):
        return np.nan
    text = str(value).upper().replace("STAGE", " ")
    match = re.search(r"\b(IV|III|II|I)\b", text)
    if match:
        return {"I": 1.0, "II": 2.0, "III": 3.0, "IV": 4.0}[match.group(1)]
    match = re.search(r"\b([1-4])\b", text)
    return float(match.group(1)) if match else np.nan


def parse_t_category(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"T([1-4])", str(value).upper())
    return float(match.group(1)) if match else np.nan


def parse_m1(value):
    if pd.isna(value):
        return np.nan
    text = str(value).upper()
    if re.search(r"M1", text):
        return 1.0
    if re.search(r"M0", text):
        return 0.0
    return np.nan


def correlation_row(cohort, endpoint, data, outcome_column):
    complete = data[["mitoxy", outcome_column]].dropna()
    rho, p_value = spearmanr(complete["mitoxy"], complete[outcome_column])
    z_value = np.arctanh(np.clip(rho, -0.999999, 0.999999))
    se = 1 / np.sqrt(len(complete) - 3)
    return {
        "cohort": cohort,
        "endpoint": endpoint,
        "n": len(complete),
        "rho": float(rho),
        "ci_low": float(np.tanh(z_value - 1.95996398454 * se)),
        "ci_high": float(np.tanh(z_value + 1.95996398454 * se)),
        "p_value": float(p_value),
    }


def read_linkedomics(path):
    matrix = pd.read_csv(path, sep="\t", index_col=0, low_memory=False)
    matrix = matrix.apply(pd.to_numeric, errors="coerce")
    matrix.index = [str(value).split("|")[0].strip().upper() for value in matrix.index]
    matrix.columns = [str(value).strip().upper() for value in matrix.columns]
    if matrix.index.duplicated().any():
        matrix = matrix.groupby(level=0, sort=False).mean()
    return matrix


def find_excel_header(path, required_terms, maximum_rows=20):
    preview = pd.read_excel(path, header=None, nrows=maximum_rows)
    best_row, best_score = None, -1
    for row_number, row in preview.iterrows():
        text = " | ".join(row.dropna().astype(str)).lower()
        score = sum(term.lower() in text for term in required_terms)
        if score > best_score:
            best_row, best_score = row_number, score
    return int(best_row)


def column_containing(columns, terms, required=True):
    for column in columns:
        normalized = str(column).strip().lower()
        if any(term.lower() in normalized for term in terms):
            return column
    if required:
        raise KeyError(f"No column matched {terms}; available: {list(columns)}")
    return None


## Public cohort reconstructions


In [ ]:
from mitoxy.clinical import primary_diagnoses
tcga_expression = pd.read_csv(
    RAW["tcga_xena_expression"], sep="\t", index_col=0
)
tcga_expression.index = tcga_expression.index.astype(str).str.upper()

missing = sorted(set(GENES) - set(tcga_expression.index))
assert not missing, f"TCGA expression missing {missing}"

tcga_gene = tcga_expression.loc[GENES].T.apply(pd.to_numeric, errors="coerce")
tcga_gene.index = tcga_gene.index.astype(str).str.upper()

# Primary-tumor samples only. Xena barcodes commonly end in ``-01``;
# some releases append an aliquot letter such as ``-01A``.
primary_mask = tcga_gene.index.str.match(
    r"^TCGA-[A-Z0-9]+-[A-Z0-9]+-01",
    na=False,
)
tcga_gene = tcga_gene.loc[primary_mask].copy()
tcga_gene["sample_id"] = tcga_gene.index.str.slice(0, 12)
tcga_gene = tcga_gene.groupby("sample_id")[GENES].mean()
assert len(tcga_gene) == 533, (
    f"Expected 533 TCGA primary-tumor cases; found {len(tcga_gene)}"
)
tcga_gene["mitoxy"] = equal4_score(tcga_gene)

clinical_object = json.loads(RAW["tcga_gdc_clinical_json"].read_text())
if isinstance(clinical_object, list):
    cases = clinical_object
elif "data" in clinical_object:
    cases = clinical_object["data"].get("hits", clinical_object["data"])
else:
    cases = clinical_object.get("hits", clinical_object)

clinical_rows = []
for case in cases:
    if not isinstance(case, dict):
        continue
    case_id = case.get("submitter_id")
    diagnoses = primary_diagnoses(case)
    grade = np.nan
    stage = np.nan
    for diagnosis in diagnoses:
        if pd.isna(grade):
            grade = parse_grade(diagnosis.get("tumor_grade"))
        if pd.isna(stage):
            stage = parse_stage(diagnosis.get("ajcc_pathologic_stage"))
        if pd.notna(grade) and pd.notna(stage):
            break
    clinical_rows.append({
        "sample_id": case_id,
        "grade": grade,
        "stage": stage,
    })

tcga_clinical = pd.DataFrame(clinical_rows).drop_duplicates("sample_id")
tcga = tcga_gene.reset_index().merge(tcga_clinical, on="sample_id", how="left")
tcga.insert(0, "cohort", "TCGA-KIRC")

print("TCGA cases:", len(tcga))
print(tcga[["grade", "stage"]].notna().sum())


In [ ]:
emtab_expression = pd.read_csv(
    RAW["emtab_expression"], sep="\t", index_col=0, low_memory=False
)

probe_map = {
    "BAK1": ["A_23_P145357"],
    # The archived analysis selected the RefSeq-mapped BAX probe below;
    # A_23_P208706 was not part of the frozen score definition.
    "BAX": ["A_33_P3341144"],
    "BID": ["A_24_P187948", "A_33_P3227041"],
    "SLC7A11": ["A_33_P3242623"],
}

biological_columns = [
    column for column in emtab_expression.columns
    if re.fullmatch(r"ccRCC-\d+", str(column), flags=re.I)
]
assert len(biological_columns) == 101

emtab_gene = pd.DataFrame(index=biological_columns)
for gene, probes in probe_map.items():
    missing_probes = sorted(set(probes) - set(emtab_expression.index.astype(str)))
    assert not missing_probes, f"Missing {gene} probes: {missing_probes}"
    emtab_gene[gene] = (
        emtab_expression.loc[probes, biological_columns]
        .apply(pd.to_numeric, errors="coerce")
        .mean(axis=0)
    )

emtab_gene.index.name = "sample_id"
emtab_gene["mitoxy"] = equal4_score(emtab_gene)

emtab_clinical_raw = pd.read_excel(
    RAW["emtab_clinical_xlsx"], header=1, dtype=str
)
emtab_clinical_raw.columns = [str(column).strip() for column in emtab_clinical_raw.columns]

emtab_clinical = pd.DataFrame({
    "sample_id": emtab_clinical_raw["sample ID"].astype(str).str.strip(),
    "sex_raw": emtab_clinical_raw["Sex"],
    "age": pd.to_numeric(emtab_clinical_raw["Age"], errors="coerce"),
    "grade": emtab_clinical_raw["Fuhrman grade"].map(parse_grade),
    "stage_raw": emtab_clinical_raw["Stage at diagnosis"],
})
emtab_clinical["stage"] = emtab_clinical["stage_raw"].map(parse_t_category)
emtab_clinical["m1"] = emtab_clinical["stage_raw"].map(parse_m1)
emtab_clinical["male"] = emtab_clinical["sex_raw"].astype(str).str.upper().eq("M").astype(float)
emtab_clinical["t34"] = np.where(
    emtab_clinical["stage"].notna(),
    (emtab_clinical["stage"] >= 3).astype(float),
    np.nan,
)

emtab = emtab_gene.reset_index().merge(emtab_clinical, on="sample_id", how="left")
emtab.insert(0, "cohort", "E-MTAB-1980")

print("E-MTAB cases:", len(emtab))
print(emtab[["grade", "stage"]].notna().sum())


In [ ]:
rds = pyreadr.read_r(str(RAW["tracerx_tpm"]))
tracerx_expression = rds[None]
tracerx_expression.index = tracerx_expression.index.astype(str).str.upper()

annotation = pd.read_csv(RAW["tracerx_annotation"], sep="\t", dtype=str)
annotation.columns = [str(column).strip() for column in annotation.columns]

sample_column = column_containing(annotation.columns, ["sample"])
patient_column = column_containing(annotation.columns, ["patient"])
type_column = column_containing(annotation.columns, ["type_collapsed", "type collapsed"])

primary_annotation = annotation.loc[
    annotation[type_column].astype(str).str.upper().eq("PRIMARY")
].copy()

primary_samples = [
    sample for sample in primary_annotation[sample_column].astype(str)
    if sample in tracerx_expression.columns
]
assert len(primary_samples) == 191

region_gene = (
    tracerx_expression.loc[GENES, primary_samples]
    .T.apply(pd.to_numeric, errors="coerce")
)
region_gene = np.log2(region_gene + 1)
region_gene["mitoxy"] = equal4_score(region_gene)
region_gene["sample_id"] = region_gene.index.astype(str)

region_gene = region_gene.merge(
    primary_annotation[[sample_column, patient_column]].rename(
        columns={sample_column: "sample_id", patient_column: "patient"}
    ),
    on="sample_id",
    how="left",
    validate="one_to_one",
)

clinical = pd.read_csv(RAW["tracerx_clinical"], sep="\t", header=1, dtype=str)
clinical.columns = [str(column).strip() for column in clinical.columns]

tracerx_clinical = pd.DataFrame({
    "patient": clinical["Subject"].astype(str).str.strip(),
    "grade": clinical["Fuhrman Grade"].map(parse_grade),
    "stage": clinical["Overall Stage"].map(parse_stage),
})

tracerx_patient = (
    region_gene.groupby("patient", as_index=False)["mitoxy"].mean()
    .merge(tracerx_clinical, on="patient", how="left")
)
tracerx_patient = tracerx_patient.rename(columns={"patient": "sample_id"})
tracerx_patient.insert(0, "cohort", "TRACERx")

# Preserve the archived direct-ID merge behavior: 73 matched clinical patients.
tracerx_patient = tracerx_patient.dropna(subset=["grade", "stage"]).copy()
assert len(tracerx_patient) == 73

print("TRACERx primary regions:", len(region_gene))
print("TRACERx clinical patients:", len(tracerx_patient))


In [ ]:
from statsmodels.duration.hazard_regression import PHReg

OUTPUT_DIR=PROJECT_ROOT/"data"/"processed"/"supplementary_figure_s1"
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

def fit_hc3(data,outcome,predictors,focal):
    complete=data[[outcome,*predictors]].replace([np.inf,-np.inf],np.nan).dropna()
    X=sm.add_constant(complete[predictors],has_constant="add").astype(float)
    fit=sm.OLS(complete[outcome].astype(float),X).fit(cov_type="HC3",use_t=False)
    ci=fit.conf_int().loc[focal]
    return {"n":len(complete),"effect":float(fit.params[focal]),"se":float(fit.bse[focal]),
            "ci_low":float(ci.iloc[0]),"ci_high":float(ci.iloc[1]),"p_value":float(fit.pvalues[focal])}

def fit_logit(data,outcome,predictors,focal):
    complete=data[[outcome,*predictors]].replace([np.inf,-np.inf],np.nan).dropna()
    X=sm.add_constant(complete[predictors],has_constant="add").astype(float)
    fit=sm.Logit(complete[outcome].astype(float),X).fit(disp=False); ci=fit.conf_int().loc[focal]
    return {"n":len(complete),"effect":float(np.exp(fit.params[focal])),"se":float(fit.bse[focal]),
            "ci_low":float(np.exp(ci.iloc[0])),"ci_high":float(np.exp(ci.iloc[1])),"p_value":float(fit.pvalues[focal])}

def fit_cox(data,duration,event,predictors,focal):
    complete=data[[duration,event,*predictors]].replace([np.inf,-np.inf],np.nan).dropna().copy()
    complete=complete.loc[complete[duration].gt(0)]
    fit=PHReg(complete[duration].astype(float),complete[predictors].astype(float),
              status=complete[event].astype(int),ties="efron").fit(disp=False)
    j=predictors.index(focal); beta=float(fit.params[j]); se=float(fit.bse[j])
    return {"n":len(complete),"events":int(complete[event].sum()),"effect":float(np.exp(beta)),"se":se,
            "ci_low":float(np.exp(beta-1.96*se)),"ci_high":float(np.exp(beta+1.96*se)),"p_value":float(fit.pvalues[j])}


## S1A — TRACERx Leibovich risk and tumor size


In [ ]:
clinical_full=pd.read_csv(RAW["tracerx_clinical"],sep="\t",header=1,dtype=str)
clinical_full.columns=[str(x).strip() for x in clinical_full.columns]
tx_extra=pd.DataFrame({
 "sample_id":clinical_full["Subject"].astype(str).str.strip(),
 "leibovich":pd.to_numeric(clinical_full["Leibovich score"],errors="coerce"),
 "tumor_size_mm":pd.to_numeric(clinical_full["Size of primary tumour (mm)"],errors="coerce")})
tx=tracerx_patient[["sample_id","mitoxy"]].merge(tx_extra,on="sample_id",how="left",validate="one_to_one")
rows=[]
for endpoint,label in [("leibovich","Leibovich score"),("tumor_size_mm","Tumor size")]:
    complete=tx[["mitoxy",endpoint]].dropna(); rho,p=spearmanr(complete.mitoxy,complete[endpoint])
    z=np.arctanh(np.clip(rho,-.999999,.999999)); se=1/np.sqrt(len(complete)-3)
    rows.append({"endpoint":label,"n":len(complete),"rho":rho,"ci_low":np.tanh(z-1.96*se),
                 "ci_high":np.tanh(z+1.96*se),"p_value":p})
panel_a=pd.DataFrame(rows); panel_a.to_csv(OUTPUT_DIR/"panel_a_tracerx_clinical.csv",index=False); display(panel_a)


## S1B–C — E-MTAB adjusted clinical and survival models


In [ ]:
# Equal4 is the mean of four gene-wise z scores, but that mean does not itself
# have unit variance. Standardize it once across the E-MTAB analysis cohort so
# all adjusted coefficients, odds ratios, and hazard ratios are per cohort SD.
emtab = emtab.copy()
emtab["mitoxy_sd"] = zscore(emtab["mitoxy"])

emtab_survival=emtab.copy()
raw_clinical=pd.read_excel(RAW["emtab_clinical_xlsx"],header=1,dtype=str)
raw_clinical.columns=[str(x).strip() for x in raw_clinical.columns]
extra=pd.DataFrame({"sample_id":raw_clinical["sample ID"].astype(str).str.strip(),
 "outcome_raw":raw_clinical["outcome"].astype(str).str.lower(),
 "os_months":pd.to_numeric(raw_clinical["observation period (month)"],errors="coerce")})
extra["death"]=np.where(extra.outcome_raw.str.contains("dead"),1.,np.where(extra.outcome_raw.str.contains("alive"),0.,np.nan))
emtab_survival=emtab_survival.merge(extra,on="sample_id",how="left",validate="one_to_one")

clinical_rows=[]
for endpoint,label in [("grade","Fuhrman grade"),("stage","T category")]:
    clinical_rows.append({"endpoint":label,"effect_type":"beta","effect_unit":"per cohort SD",
                          **fit_hc3(emtab,endpoint,["mitoxy_sd","age","male"],"mitoxy_sd")})
for endpoint,label in [("t34","T3–T4 disease"),("m1","M1 disease")]:
    clinical_rows.append({"endpoint":label,"effect_type":"odds ratio","effect_unit":"per cohort SD",
                          **fit_logit(emtab,endpoint,["mitoxy_sd","age","male"],"mitoxy_sd")})
panel_b=pd.DataFrame(clinical_rows)
panel_b.to_csv(OUTPUT_DIR/"panel_b_emtab_adjusted.csv",index=False)
display(panel_b)

survival_rows=[]
for model,predictors in [("Unadjusted",["mitoxy_sd"]),("Age + sex",["mitoxy_sd","age","male"]),
                         ("Grade + TNM adjusted",["mitoxy_sd","age","male","grade","stage","m1"])]:
    survival_rows.append({"model":model,"effect_type":"hazard ratio","effect_unit":"per cohort SD",
                          **fit_cox(emtab_survival,"os_months","death",predictors,"mitoxy_sd")})
panel_c=pd.DataFrame(survival_rows)
panel_c.to_csv(OUTPUT_DIR/"panel_c_emtab_survival.csv",index=False)
display(panel_c)


## S1D — corresponding TCGA survival sensitivity


In [ ]:
from mitoxy.clinical import primary_diagnoses
clinical_object = json.loads(
    RAW["tcga_gdc_clinical_json"].read_text()
)

cases = clinical_object.get(
    "data", {}
).get(
    "hits",
    clinical_object if isinstance(clinical_object, list) else [],
)


def first_numeric(values):
    for value in values:
        numeric = pd.to_numeric(value, errors="coerce")
        if pd.notna(numeric):
            return float(numeric)
    return np.nan


def first_parsed(values, parser):
    for value in values:
        parsed = parser(value)
        if pd.notna(parsed):
            return float(parsed)
    return np.nan


rows = []

for case in cases:
    if not isinstance(case, dict):
        continue

    demographic = case.get("demographic") or {}
    diagnoses = primary_diagnoses(case)

    vital_status = str(
        demographic.get("vital_status", "")
    ).strip().casefold()

    if vital_status == "dead":
        death = 1.0
    elif vital_status == "alive":
        death = 0.0
    else:
        death = np.nan

    days_to_death = pd.to_numeric(
        demographic.get("days_to_death"),
        errors="coerce",
    )

    follow_up_values = [
        pd.to_numeric(
            diagnosis.get("days_to_last_follow_up"),
            errors="coerce",
        )
        for diagnosis in diagnoses
    ]
    follow_up_values = [
        value for value in follow_up_values
        if pd.notna(value)
    ]
    days_to_follow_up = (
        max(follow_up_values)
        if follow_up_values
        else np.nan
    )

    if death == 1.0 and pd.notna(days_to_death):
        os_days = float(days_to_death)
    else:
        os_days = days_to_follow_up

    age_days = first_numeric([
        diagnosis.get("age_at_diagnosis")
        for diagnosis in diagnoses
    ])

    if pd.notna(age_days):
        age = age_days / 365.25
    else:
        # Current GDC responses also provide age_at_index in years.
        age = pd.to_numeric(
            demographic.get("age_at_index"),
            errors="coerce",
        )

    # Current GDC schema uses sex_at_birth; older responses used gender.
    sex_value = demographic.get("sex_at_birth")
    if sex_value is None:
        sex_value = demographic.get("gender")

    sex_text = str(sex_value).strip().casefold()

    if sex_text == "male":
        male = 1.0
    elif sex_text == "female":
        male = 0.0
    else:
        male = np.nan

    grade = first_parsed(
        [
            diagnosis.get("tumor_grade")
            for diagnosis in diagnoses
        ],
        parse_grade,
    )

    stage = first_parsed(
        [
            diagnosis.get("ajcc_pathologic_stage")
            for diagnosis in diagnoses
        ],
        parse_stage,
    )

    rows.append({
        "sample_id": case.get("submitter_id"),
        "os_days": os_days,
        "death": death,
        "age": age,
        "male": male,
        "grade": grade,
        "stage": stage,
    })


tcga_survival_score = tcga[["sample_id", "mitoxy"]].copy()
tcga_survival_score["mitoxy_sd"] = zscore(
    tcga_survival_score["mitoxy"]
)

tcga_survival = (
    tcga_survival_score[["sample_id", "mitoxy_sd"]]
    .merge(
        pd.DataFrame(rows).drop_duplicates("sample_id"),
        on="sample_id",
        how="left",
        validate="one_to_one",
    )
)


# Redefine the Cox helper so a non-estimable nuisance variable cannot make
# the entire model fail. The focal Mitoxy variable is always retained.
def fit_cox(data, duration, event, predictors, focal):
    complete = (
        data[[duration, event, *predictors]]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .copy()
    )
    complete = complete.loc[
        complete[duration].gt(0)
    ].copy()

    if len(complete) == 0:
        raise RuntimeError(
            f"No complete observations for Cox model: {predictors}"
        )

    if complete[event].sum() == 0:
        raise RuntimeError(
            f"No events in Cox model: {predictors}"
        )

    if complete[focal].nunique() < 2:
        raise RuntimeError(
            f"Focal predictor {focal!r} has no variation"
        )

    active_predictors = [focal]

    for predictor in predictors:
        if predictor == focal:
            continue

        if complete[predictor].nunique() < 2:
            warnings.warn(
                f"Dropping constant Cox nuisance variable: {predictor}"
            )
            continue

        candidate = active_predictors + [predictor]
        design = complete[candidate].astype(float).to_numpy()

        if np.linalg.matrix_rank(design) < len(candidate):
            warnings.warn(
                f"Dropping collinear Cox nuisance variable: {predictor}"
            )
            continue

        active_predictors.append(predictor)

    fit = PHReg(
        complete[duration].astype(float),
        complete[active_predictors].astype(float),
        status=complete[event].astype(int),
        ties="efron",
    ).fit(disp=False)

    focal_position = active_predictors.index(focal)
    beta = float(fit.params[focal_position])
    standard_error = float(fit.bse[focal_position])

    return {
        "n": len(complete),
        "events": int(complete[event].sum()),
        "effect": float(np.exp(beta)),
        "se": standard_error,
        "ci_low": float(np.exp(beta - 1.96 * standard_error)),
        "ci_high": float(np.exp(beta + 1.96 * standard_error)),
        "p_value": float(fit.pvalues[focal_position]),
    }


print("TCGA survival-variable audit:")
display(
    pd.DataFrame({
        "nonmissing": tcga_survival[
            [
                "os_days",
                "death",
                "mitoxy_sd",
                "age",
                "male",
                "grade",
                "stage",
            ]
        ].notna().sum(),
        "unique": tcga_survival[
            [
                "os_days",
                "death",
                "mitoxy_sd",
                "age",
                "male",
                "grade",
                "stage",
            ]
        ].nunique(dropna=True),
    })
)


cox = []

models = [
    (
        "Unadjusted",
        ["mitoxy_sd"],
    ),
    (
        "Age + sex",
        ["mitoxy_sd", "age", "male"],
    ),
    (
        "Grade + stage adjusted",
        ["mitoxy_sd", "age", "male", "grade", "stage"],
    ),
]

for model, predictors in models:
    cox.append({
        "model": model,
        "effect_type": "hazard ratio",
        "effect_unit": "per cohort SD",
        **fit_cox(
            tcga_survival,
            "os_days",
            "death",
            predictors,
            "mitoxy_sd",
        ),
    })

panel_d = pd.DataFrame(cox)
panel_d.to_csv(
    OUTPUT_DIR / "panel_d_tcga_survival.csv",
    index=False,
)

display(panel_d)


In [ ]:
contracts = {'panel_a_tracerx_clinical.csv': ['endpoint', 'n', 'rho', 'ci_low', 'ci_high', 'p_value'], 'panel_b_emtab_adjusted.csv': ['endpoint', 'effect_type', 'n', 'effect', 'ci_low', 'ci_high', 'p_value'], 'panel_c_emtab_survival.csv': ['model', 'n', 'events', 'effect', 'ci_low', 'ci_high', 'p_value'], 'panel_d_tcga_survival.csv': ['model', 'n', 'events', 'effect', 'ci_low', 'ci_high', 'p_value']}
rows=[]
for filename,required in contracts.items():
    path=OUTPUT_DIR/filename
    table=pd.read_csv(path)
    missing=sorted(set(required)-set(table.columns))
    if missing: raise KeyError(f"{filename} missing {missing}")
    rows.append({"file":filename,"rows":len(table),"size_bytes":path.stat().st_size,"sha256":sha256_file(path)})
processed_manifest=pd.DataFrame(rows)
processed_manifest.to_csv(OUTPUT_DIR/"processed_manifest.csv",index=False)
display(processed_manifest)
print("Processed-file handoff: PASS")
